In [ ]:
# ============================================================
# SERIAL VS PARALLEL BINARY ADDER
# ============================================================
#
# This notebook compares the operation and performance of a
# serial binary adder with those of a parallel ripple-carry adder.
#
# HOW TO USE THE NOTEBOOK
#
# 1. Use the "Word length N" slider to select the number of bits
#    contained in each binary word.
#
# 2. Use the "Clock frequency" slider to select the operating
#    clock frequency of the digital system.
#
# 3. Use the "Processing step" slider to follow the processing
#    of the N bit positions.
#
# 4. In the SERIAL ADDER:
#
#       - only one pair of bits is processed during each
#         clock cycle,
#       - therefore an N-bit addition requires N clock cycles,
#       - the theoretical throughput is f/N additions per second.
#
# 5. In the PARALLEL ADDER:
#
#       - N full adders process the N pairs of bits simultaneously,
#       - one complete addition can ideally be performed during
#         each clock cycle,
#       - the theoretical throughput is f additions per second.
#
# 6. The parallel ripple-carry diagram also demonstrates the
#    propagation of the carry from one full-adder stage to the
#    next.
#
# 7. Increasing N makes the serial implementation progressively
#    slower, while the parallel implementation requires
#    progressively more hardware.
#
# IMPORTANT
#
# The throughput expressions f/N and f correspond to the simplified
# comparison presented in the theory. In an actual parallel adder,
# the maximum usable clock frequency is also constrained by the
# propagation delay of the carry chain and the logic gates.
#
# ============================================================


from ipywidgets import IntSlider, FloatSlider, HBox, VBox, Layout, HTML
from IPython.display import display


# ------------------------------------------------------------
# Global style sheet
# ------------------------------------------------------------

style_html = HTML("""
<style>

.adder-root {
    font-family: monospace;
    width: 100%;
    max-width: 900px;
    box-sizing: border-box;
}

.adder-title {
    font-size: 22px;
    font-weight: bold;
    margin-bottom: 8px;
    color: #1f1f1f;
}

.description-box {
    font-size: 13px;
    line-height: 1.45;
    padding: 9px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 9px;
    box-sizing: border-box;
    white-space: normal;
}

.section-box {
    border: 1px solid #c8d0dc;
    border-radius: 10px;
    padding: 10px 12px;
    background: #ffffff;
    box-sizing: border-box;
    width: 100%;
}

.section-title {
    font-size: 16px;
    font-weight: bold;
    margin-bottom: 7px;
    color: #243447;
}

.bit-line {
    white-space: normal;
    margin-top: 6px;
    margin-bottom: 6px;
    line-height: 36px;
    max-width: 100%;
}

.bit-box {
    display: inline-block;
    width: 25px;
    height: 29px;
    line-height: 29px;
    text-align: center;
    margin-right: 3px;
    border-radius: 5px;
    border: 1px solid #7f8c9a;
    font-size: 13px;
    font-weight: bold;
    box-sizing: border-box;
}

.bit-idle {
    background: #f0f2f4;
    color: #68717c;
}

.bit-active {
    background: #fff3bf;
    color: #111111;
    border: 3px solid #d62828;
}

.bit-done {
    background: #dff3e4;
    color: #1d5d2d;
}

.stage-box {
    display: inline-block;
    width: 25px;
    height: 29px;
    line-height: 29px;
    text-align: center;
    margin-right: 2px;
    border-radius: 5px;
    border: 1px solid #7f8c9a;
    font-size: 11px;
    font-weight: bold;
    box-sizing: border-box;
}

.stage-ready {
    background: #dceeff;
    color: #0e3a66;
}

.stage-carry {
    background: #fff3bf;
    color: #111111;
    border: 3px solid #d62828;
}

.stage-complete {
    background: #dff3e4;
    color: #1d5d2d;
}

.arrow {
    display: inline-block;
    font-size: 15px;
    margin-right: 2px;
    color: #555555;
}

.info {
    font-size: 14px;
    line-height: 1.6;
}

.metric {
    display: inline-block;
    min-width: 185px;
    font-weight: bold;
    color: #243447;
}

.serial-color {
    color: #9b3d00;
}

.parallel-color {
    color: #165a31;
}

.small-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    white-space: normal;
    margin-top: 5px;
}

.controls-title {
    font-family: monospace;
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.comparison-table {
    border-collapse: collapse;
    font-family: monospace;
    font-size: 13px;
    width: 100%;
    line-height: 1.2;
}

.comparison-table th,
.comparison-table td {
    border: 1px solid #c8d0dc;
    padding: 4px 8px;
    text-align: center;
    vertical-align: middle;
}

.comparison-table th {
    background: #f2f4f7;
}

</style>
""")


# ------------------------------------------------------------
# HTML containers
# ------------------------------------------------------------

title_html = HTML("""
<div class="adder-root">
    <div class="adder-title">
        Serial vs Parallel Binary Adder
    </div>
</div>
""")


description_html = HTML("""
<div class="adder-root">

    <div class="description-box">

        A <b>serial adder</b> uses one full-adder stage and processes one pair of bits during each clock cycle.
        Therefore, an N-bit addition requires N clock cycles.<br>

        A <b>parallel adder</b> uses N full-adder stages so that all pairs of bits can be processed simultaneously.
        This greatly increases speed, but also increases hardware complexity.<br>

        Use the <b>Processing step</b> slider to compare the sequential operation of the serial adder
        with the propagation of the carry through a parallel ripple-carry architecture.

    </div>

</div>
""")


performance_html = HTML()
serial_html = HTML()
parallel_html = HTML()
comparison_html = HTML()


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(width='285px')

style_opts = {'description_width': '115px'}


n_slider = IntSlider(
    min=2,
    max=16,
    step=1,
    value=8,
    description='Word length N:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


frequency_slider = FloatSlider(
    min=1.0,
    max=100.0,
    step=1.0,
    value=20.0,
    description='Clock frequency:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True,
    readout_format='.1f'
)


step_slider = IntSlider(
    min=1,
    max=n_slider.value,
    step=1,
    value=1,
    description='Processing step:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


controls_title = HTML("""
<div class="controls-title">
Controls
</div>
""")


controls_box = VBox(
    [
        controls_title,
        n_slider,
        frequency_slider,
        step_slider
    ],
    layout=Layout(
        width='315px',
        min_width='315px',
        border='1px solid #c8d0dc',
        padding='10px',
        overflow='visible'
    )
)


# ------------------------------------------------------------
# Visual helpers
# ------------------------------------------------------------

def render_serial_bits(N, step):
    boxes = []

    for position in range(N):
        if position < step - 1:
            css = "bit-done"
        elif position == step - 1:
            css = "bit-active"
        else:
            css = "bit-idle"

        boxes.append(f"<span class='bit-box {css}'>{position}</span>")

    return ''.join(boxes)


def render_parallel_stages(N, step):
    items = []

    for position in range(N):
        if position < step - 1:
            css = "stage-complete"
        elif position == step - 1:
            css = "stage-carry"
        else:
            css = "stage-ready"

        items.append(f"<span class='stage-box {css}'>FA</span>")

        if position < N - 1:
            items.append("<span class='arrow'>→</span>")

    return ''.join(items)


# ------------------------------------------------------------
# Main update function
# ------------------------------------------------------------

def update_display(*args):
    N = n_slider.value
    f_MHz = frequency_slider.value
    step = step_slider.value

    f_Hz = f_MHz * 1e6
    clock_period = 1.0 / f_Hz

    serial_cycles = N
    parallel_cycles = 1

    serial_time = N * clock_period
    parallel_time = clock_period

    serial_throughput = f_Hz / N
    parallel_throughput = f_Hz

    speedup = parallel_throughput / serial_throughput

    serial_time_us = serial_time * 1e6
    parallel_time_us = parallel_time * 1e6

    serial_throughput_M = serial_throughput / 1e6
    parallel_throughput_M = parallel_throughput / 1e6


    performance_html.value = f"""
    <div class="section-box">

        <div class="section-title">
            Performance Summary
        </div>

        <div class="info">

            <span class="metric">Word length</span>
            N = {N} bits
            <br>

            <span class="metric">Clock frequency</span>
            f = {f_MHz:.1f} MHz
            <br>

            <span class="metric">Clock period</span>
            T<sub>clk</sub> = {clock_period * 1e9:.2f} ns

        </div>

        <table class="comparison-table" style="margin-top:7px;">

            <tr>
                <th>Quantity</th>
                <th>Serial adder</th>
                <th>Parallel adder</th>
            </tr>

            <tr>
                <td>Clock cycles / addition</td>
                <td>{serial_cycles}</td>
                <td>{parallel_cycles}</td>
            </tr>

            <tr>
                <td>Time / addition</td>
                <td>{serial_time_us:.4f} μs</td>
                <td>{parallel_time_us:.4f} μs</td>
            </tr>

            <tr>
                <td>Throughput</td>
                <td>{serial_throughput_M:.4f} M additions/s</td>
                <td>{parallel_throughput_M:.4f} M additions/s</td>
            </tr>

            <tr>
                <td>Ideal throughput expression</td>
                <td>f / N</td>
                <td>f</td>
            </tr>

        </table>

        <div class="small-note">
            For the same clock frequency, the ideal throughput improvement
            of the parallel architecture is a factor of <b>{speedup:.0f}</b>.
        </div>

    </div>
    """


    serial_html.value = f"""
    <div class="section-box">

        <div class="section-title serial-color">
            Serial Adder
        </div>

        <div class="info">
            One pair of bits is processed during each clock cycle.
        </div>

        <div class="bit-line">
            {render_serial_bits(N, step)}
        </div>

        <div class="small-note">
            Each box represents one bit position, starting with the LSB at position 0.
            The yellow box is processed during the current clock cycle, while green boxes have already been processed.
        </div>

        <div class="info" style="margin-top:7px;">

            <span class="metric">Current cycle</span>
            {step} of {N}
            <br>

            <span class="metric">Current bit position</span>
            {step - 1}
            <br>

            <span class="metric">Completed fraction</span>
            {100.0 * step / N:.1f} %

        </div>

    </div>
    """


    parallel_html.value = f"""
    <div class="section-box">

        <div class="section-title parallel-color">
            Parallel Ripple-Carry Adder
        </div>

        <div class="info">
            All N full adders are present simultaneously.
        </div>

        <div style="
            white-space: normal;
            margin-top:7px;
            margin-bottom:7px;
            line-height:36px;
        ">
            {render_parallel_stages(N, step)}
        </div>

        <div class="small-note">
            The yellow full-adder stage indicates the current position reached by the carry.
            Green stages have already received and propagated the carry, while blue stages are still farther along the carry chain.
        </div>

        <div class="info" style="margin-top:7px;">

            <span class="metric">Carry stage</span>
            {step} of {N}
            <br>

            <span class="metric">Full adders required</span>
            {N}
            <br>

            <span class="metric">Clock-level throughput</span>
            f = {f_MHz:.1f} M additions/s

        </div>

    </div>
    """


    comparison_html.value = f"""
    <div class="section-box">

        <div class="section-title">
            Main Trade-Off
        </div>

        <div class="info">

            <span class="serial-color"><b>Serial architecture:</b></span>
            simple hardware and low implementation cost, but an N-bit addition requires N clock cycles.<br>

            <span class="parallel-color"><b>Parallel architecture:</b></span>
            much higher throughput, but approximately N full-adder stages are required.<br>

            Therefore, as N increases:
            <b>Serial throughput = f / N</b> decreases, while
            <b>Parallel throughput = f</b> remains one complete addition per clock cycle in the simplified model.

        </div>

        <div class="small-note">
            The ripple-carry visualization illustrates an important practical limitation:
            the carry must propagate through successive full-adder stages.
            Therefore, a real parallel adder cannot increase its clock frequency
            without regard to gate propagation delays.
        </div>

    </div>
    """


# ------------------------------------------------------------
# Reset processing step when N changes
# ------------------------------------------------------------

def update_word_length(change):
    N = change['new']

    step_slider.max = N

    if step_slider.value != 1:
        step_slider.value = 1
    else:
        update_display()


n_slider.observe(update_word_length, names='value')
frequency_slider.observe(update_display, names='value')
step_slider.observe(update_display, names='value')


# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

performance_html.layout = Layout(
    width='565px',
    min_width='565px',
    overflow='visible'
)


top_row = HBox(
    [
        performance_html,
        controls_box
    ],
    layout=Layout(
        width='900px',
        max_width='900px',
        align_items='flex-start',
        justify_content='flex-start',
        overflow='visible'
    )
)


serial_html.layout = Layout(
    width='430px',
    min_width='430px',
    overflow='visible'
)


parallel_html.layout = Layout(
    width='450px',
    min_width='450px',
    overflow='visible'
)


architecture_row = HBox(
    [
        serial_html,
        parallel_html
    ],
    layout=Layout(
        width='900px',
        max_width='900px',
        align_items='flex-start',
        justify_content='space-between',
        overflow='visible'
    )
)


comparison_html.layout = Layout(
    width='900px',
    max_width='900px',
    overflow='visible'
)


main_layout = VBox(
    [
        top_row,
        architecture_row,
        comparison_html
    ],
    layout=Layout(
        width='900px',
        max_width='900px',
        align_items='flex-start',
        overflow='visible'
    )
)


# ------------------------------------------------------------
# Initial display
# ------------------------------------------------------------

update_display()

display(style_html)
display(title_html)
display(description_html)
display(main_layout)